# 01 — Fake News Detection : EDA & Preprocessing
## Objectif : comprendre le dataset avant de modéliser
### Ce notebook produit : données nettoyées + splits train/val/test

**cellule 1** : installation des dépendances

In [ ]:
# Monte Drive — à exécuter en premier dans chaque notebook
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/FakeNewsProject'

os.makedirs(f'{BASE}/data/raw',       exist_ok=True)
os.makedirs(f'{BASE}/data/processed', exist_ok=True)
os.makedirs(f'{BASE}/models/bert_final', exist_ok=True)
os.makedirs(f'{BASE}/figures',        exist_ok=True)
os.makedirs(f'{BASE}/preds',          exist_ok=True)

print(f"✓ Drive monté — BASE = {BASE}")

In [ ]:
# À exécuter en premier — installation unique
!pip install -q kaggle pandas numpy matplotlib seaborn \
    scikit-learn spacy wordcloud plotly mlflow
!python -m spacy download en_core_web_sm -q

import warnings
warnings.filterwarnings('ignore')
print("✓ Dépendances installées")

**cellule 2** : importation Kaggle

In [ ]:
import os


os.environ['KAGGLE_TOKEN'] = 'KGAT_86bb58c8aa99a03064efc4b25c08cddc'

# Créer le kaggle.json manuellement
os.makedirs('/root/.kaggle', exist_ok=True)

# Mon username Kaggle
kaggle_json = {
    "username": "malakben204",
    "key": os.environ['KAGGLE_TOKEN']
}

import json
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_json, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Télécharger le dataset
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset
!unzip -q fake-and-real-news-dataset.zip -d data/
!ls data/

print("✓ Dataset prêt")

**cellule 3** : Chargement et fusion des données

In [ ]:
import pandas as pd
import numpy as np

# Charger les deux fichiers
fake_df = pd.read_csv('data/Fake.csv')
real_df = pd.read_csv('data/True.csv')

# Ajouter les labels AVANT de fusionner
fake_df['label'] = 0  # 0 = FAKE
real_df['label']  = 1  # 1 = REAL

# Fusionner et mélanger (important !)
df = pd.concat([fake_df, real_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset total : {len(df):,} articles")
print(f"Colonnes : {list(df.columns)}")
print("\nDistribution des labels :")
print(df['label'].value_counts())
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")
df.head(3)

**cellule 4** : EDA visualisation clé

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Fake News Dataset — Analyse exploratoire',
             fontsize=16, fontweight='bold', y=1.02)

# 1. Distribution des labels
label_counts = df['label'].value_counts()
axes[0,0].pie(label_counts, labels=['Fake','Real'],
              autopct='%1.1f%%', colors=['#E24B4A','#1D9E75'],
              startangle=90)
axes[0,0].set_title('Distribution Fake vs Real')

# 2. Longueur des textes par label
df['text_len'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

for label, color, name in [(0,'#E24B4A','Fake'),(1,'#1D9E75','Real')]:
    axes[0,1].hist(
        df[df['label']==label]['word_count'].clip(0,2000),
        bins=50, alpha=0.6, color=color, label=name
    )
axes[0,1].set_title('Distribution du nombre de mots')
axes[0,1].legend()
axes[0,1].set_xlabel('Nombre de mots (capped à 2000)')

# 3. Top sujets par label
fake_subjects = df[df['label']==0]['subject'].value_counts().head(6)
real_subjects = df[df['label']==1]['subject'].value_counts().head(6)

axes[0,2].barh(fake_subjects.index, fake_subjects.values, color='#E24B4A')
axes[0,2].set_title('Top sujets — FAKE')

axes[1,0].barh(real_subjects.index, real_subjects.values, color='#1D9E75')
axes[1,0].set_title('Top sujets — REAL')

# 4. Stats descriptives
stats = df.groupby('label')['word_count'].describe()[
    ['mean','std','min','max']
].round(0)
axes[1,1].axis('off')
table = axes[1,1].table(
    cellText=stats.values, rowLabels=['Fake','Real'],
    colLabels=stats.columns, loc='center', cellLoc='center'
)
table.scale(1, 2)
axes[1,1].set_title('Stats — nombre de mots')

# 5. Boxplot word count
df['label_str'] = df['label'].map({0:'Fake',1:'Real'})
df[df['word_count']<2000].boxplot(
    column='word_count', by='label_str', ax=axes[1,2]
)
axes[1,2].set_title('Boxplot word count')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

**cellule 5** : Détection du Dataleakage

In [ ]:
# PROBLÈME CONNU de ce dataset : la colonne 'subject' trahit le label
# Si on inclut 'subject' dans les features → data leakage → 99.9% accuracy FAUSSE

print("=== AUDIT DATA LEAKAGE ===")
print("\nSujets exclusifs aux FAKE news :")
fake_only = set(df[df['label']==0]['subject'].unique()) - \
            set(df[df['label']==1]['subject'].unique())
print(fake_only)

print("\nSujets exclusifs aux REAL news :")
real_only = set(df[df['label']==1]['subject'].unique()) - \
            set(df[df['label']==0]['subject'].unique())
print(real_only)

# Vérifier aussi le champ 'date' — certains patterns de date peuvent être discriminants
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print("\nPériode FAKE :", df[df['label']==0]['date'].agg(['min','max']))
print("Période REAL :", df[df['label']==1]['date'].agg(['min','max']))

# CONCLUSION : on utilise UNIQUEMENT title + text comme features
print("\n✓ Features retenues : title + text uniquement")
print("✗ Features exclues : subject, date (data leakage)")

**cellule 6** : Preprocessing NLP - Pipeline complet

In [ ]:
import re
import spacy
from tqdm.auto import tqdm
tqdm.pandas()

nlp = spacy.load("en_core_web_sm", disable=["parser","ner"])

def clean_text(text: str) -> str:
    """Pipeline de nettoyage NLP robuste."""
    if not isinstance(text, str):
        return ""

    # 1. Minuscules
    text = text.lower()

    # 2. Supprimer Reuters/AP datelines (bruit spécifique au dataset)
    text = re.sub(r'^.*?\(reuters\)\s*-?\s*', '', text)
    text = re.sub(r'^.*?\(ap\)\s*-?\s*', '', text)

    # 3. URLs, emails, chiffres
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)
    text = re.sub(r'\d+', ' NUM ', text)

    # 4. Ponctuation (garder les apostrophes)
    text = re.sub(r"[^a-zA-Z\s']", ' ', text)

    # 5. Lemmatisation + suppression stopwords via spaCy
    doc = nlp(text[:100000])  # limite pour la RAM
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop
        and not token.is_punct
        and len(token.text) > 2
    ]

    return " ".join(tokens)

# Combiner title + text (le titre est souvent très discriminant)
df['full_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

# Appliquer le preprocessing (prend ~5 min sur Colab)
print("Nettoyage en cours...")
df['clean_text'] = df['full_text'].progress_apply(clean_text)

# Pour BERT : version non-nettoyée (BERT gère lui-même le contexte)
df['bert_text'] = (df['title'].fillna('') + ' [SEP] ' +
                   df['text'].fillna('').str[:500])

print("✓ Preprocessing terminé")
print(f"\nExemple avant : {df['full_text'].iloc[0][:100]}...")
print(f"Exemple après : {df['clean_text'].iloc[0][:100]}...")

**cellule 7** : Wordcloud - Visualisation des mots clés

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# Check if df is defined. If not, raise an informative error.
if 'df' not in globals() and 'df' not in locals():
    raise NameError("DataFrame 'df' is not defined. Please ensure all preceding cells, especially those loading and preprocessing the data (e.g., cell 'Co4NVfwBvgr-' and 'A0ej5Uhvvtu-'), have been executed.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for ax, label, title, cmap in [
    (ax1, 0, 'Mots les plus fréquents — FAKE', 'Reds'),
    (ax2, 1, 'Mots les plus fréquents — REAL', 'Greens')
]:
    text_corpus = " ".join(df[df['label']==label]['clean_text'])

    wc = WordCloud(
        width=800, height=400,
        background_color='white', colormap=cmap,
        max_words=100, collocations=False
    ).generate(text_corpus)

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150)
plt.show()

**cellule 8** : Train / Val / Test

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['clean_text', 'bert_text', 'full_text']]
y = df['label']

# Split 1 : 80% trainval / 20% test (test = intouchable)
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Split 2 : 75% train / 25% val (soit 60/20/20 au total)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)

print("=== SPLITS ===")
for name, X_, y_ in [
    ("Train", X_train, y_train),
    ("Val",   X_val,   y_val),
    ("Test",  X_test,  y_test)
]:
    n_fake = (y_==0).sum()
    n_real = (y_==1).sum()
    print(f"{name:6} : {len(y_):5} ({n_fake} fake | {n_real} real) — "
          f"ratio fake: {n_fake/len(y_):.1%}")

# Sauvegarder les splits pour les notebooks suivants
import os
os.makedirs('data/processed', exist_ok=True)

train_df = X_train.copy(); train_df['label'] = y_train
val_df   = X_val.copy();   val_df['label']   = y_val
test_df  = X_test.copy();  test_df['label']  = y_test

train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv',     index=False)
test_df.to_csv('data/processed/test.csv',   index=False)

print("\n✓ Splits sauvegardés dans data/processed/")
print("✓ Notebook 01 terminé — passer au Notebook 02")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/drive/MyDrive/FakeNewsProject/data/processed', exist_ok=True)
os.makedirs('/content/drive/MyDrive/FakeNewsProject/figures', exist_ok=True)

# Copier les données
shutil.copy('data/processed/train.csv', '/content/drive/MyDrive/FakeNewsProject/data/processed/')
shutil.copy('data/processed/val.csv',   '/content/drive/MyDrive/FakeNewsProject/data/processed/')
shutil.copy('data/processed/test.csv',  '/content/drive/MyDrive/FakeNewsProject/data/processed/')

# Copier les figures
shutil.copy('eda_overview.png', '/content/drive/MyDrive/FakeNewsProject/figures/')
shutil.copy('wordclouds.png',   '/content/drive/MyDrive/FakeNewsProject/figures/')

print("✓ Tout sauvegardé sur Drive")